# 01. Exploratory Data Analysis & ABC Inventory Segmentation
## Supply Chain Demand Forecasting & Inventory Optimization Project

### Executive & Business Objective
> **Target Business Question:**
> *"For our top 20 SKUs by revenue, what should the reorder point and safety stock be for the next quarter, given demand variability and current supplier lead times — and which SKUs are at highest stockout risk?"*

In this notebook, we perform end-to-end exploratory analysis on our cleaned supply chain transaction data:
1. Verify database integrity and clean transaction volume.
2. Check temporal continuity (missing weeks, outlier detection).
3. Analyze demand trends, volatility, and category-level seasonality.
4. Perform **ABC Analysis (Pareto 80/15/5 Rule)** to segment high-value SKUs and isolate the Top 20 revenue drivers.


In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DB_PATH = os.path.join(BASE_DIR, "data", "supply_chain.db")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

print(f"Connecting to database: {DB_PATH}")


### 1. Database Connection & Data Integrity Summary
Let's connect to `supply_chain.db` (populated via our SQL pipeline) and inspect the volume of cleaned orders and suppliers.


In [ ]:
conn = sqlite3.connect(DB_PATH)

summary_df = pd.read_sql_query('''
    SELECT 
        COUNT(*) AS total_clean_orders,
        COUNT(DISTINCT order_id) AS distinct_orders,
        COUNT(DISTINCT sku_id) AS distinct_skus,
        COUNT(DISTINCT customer_id) AS distinct_customers,
        MIN(order_date) AS min_date,
        MAX(order_date) AS max_date,
        ROUND(SUM(line_total_revenue), 2) AS total_revenue
    FROM cleaned_orders;
''', conn)

summary_df


: 

### 2. Temporal Continuity & Weekly Demand Aggregation
Let's verify weekly demand patterns and ensure no missing weeks exist in the 104-week historical period (2023-01 to 2024-12).


In [3]:
weekly_df = pd.read_sql_query('''
    SELECT 
        sku_id,
        product_name,
        category,
        week_start,
        units_sold,
        weekly_revenue
    FROM v_weekly_demand_by_sku
    ORDER BY sku_id, week_start;
''', conn)

weekly_df['week_start'] = pd.to_datetime(weekly_df['week_start'])
print(f"Total SKU-week records: {len(weekly_df):,}")
print(f"Date range: {weekly_df['week_start'].min().date()} to {weekly_df['week_start'].max().date()}")

weeks_per_sku = weekly_df.groupby('sku_id')['week_start'].nunique()
print(f"Min weeks per SKU: {weeks_per_sku.min()}, Max weeks per SKU: {weeks_per_sku.max()}")
assert weeks_per_sku.min() == 104, "Data has missing weeks! Complete timeline required."
print("Integrity Check Passed: All 50 SKUs possess 104 continuous weeks of transactional demand.")


### 3. ABC Inventory Segmentation (Pareto 80/20 Rule)
Supply chain managers segment catalog items using **ABC Analysis**:
- **Class A (Top ~80% of Total Revenue):** High-priority items requiring stringent service levels, dynamic safety stock, and continuous monitoring.
- **Class B (~15% of Total Revenue):** Moderate-value items managed via periodic reviews.
- **Class C (Bottom ~5% of Total Revenue):** Long-tail, low-value items where simple bulk ordering minimizes overhead.


In [4]:
abc_df = pd.read_sql_query('''
    SELECT * FROM v_sku_abc_classification;
''', conn)

print("ABC Classification Breakdown:")
abc_summary = abc_df.groupby('abc_category').agg(
    sku_count=('sku_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    total_units=('total_units', 'sum')
).reset_index()

grand_revenue = abc_summary['total_revenue'].sum()
abc_summary['revenue_pct'] = (abc_summary['total_revenue'] / grand_revenue) * 100
abc_summary['sku_pct'] = (abc_summary['sku_count'] / len(abc_df)) * 100

abc_summary


### 4. Visualizing the Pareto Curve
Let's visualize the cumulative revenue distribution across all 50 SKUs to clearly illustrate the Pareto principle.


In [5]:
fig, ax1 = plt.subplots(figsize=(14, 6))

x = np.arange(len(abc_df))
ax1.bar(x, abc_df['total_revenue'] / 1000.0, color='#1F4E79', alpha=0.85, label='SKU Revenue ($k)')
ax1.set_xlabel('SKU Rank (Descending Revenue)')
ax1.set_ylabel('SKU Total Revenue ($k)', color='#1F4E79')
ax1.tick_params(axis='y', labelcolor='#1F4E79')

ax2 = ax1.twinx()
ax2.plot(x, abc_df['cumulative_revenue_pct'], color='#C00000', linewidth=2.5, label='Cumulative Revenue %')
ax2.axhline(80, color='orange', linestyle='--', label='80% Pareto Threshold (Class A)')
ax2.axhline(95, color='gray', linestyle=':', label='95% Threshold (Class B)')
ax2.set_ylabel('Cumulative Revenue (%)', color='#C00000')
ax2.tick_params(axis='y', labelcolor='#C00000')
ax2.set_ylim(0, 105)

plt.title('ABC Inventory Segmentation: Pareto Revenue Distribution (50 SKUs)', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()


### 5. Demand Trend & Seasonality for Top Revenue SKUs
Let's inspect weekly sales trends for our top 4 revenue drivers to observe their seasonality profiles (e.g. Q4 holiday peaks, summer spikes, new year fitness surges).


In [6]:
top_4_skus = abc_df.head(4)['sku_id'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
axes = axes.flatten()

for idx, sku in enumerate(top_4_skus):
    sku_data = weekly_df[weekly_df['sku_id'] == sku]
    sku_name = sku_data['product_name'].iloc[0]
    cat = sku_data['category'].iloc[0]
    
    ax = axes[idx]
    ax.plot(sku_data['week_start'], sku_data['units_sold'], color='#1F4E79', lw=1.8, label='Weekly Units Sold')
    rolling_avg = sku_data['units_sold'].rolling(window=8, min_periods=1).mean()
    ax.plot(sku_data['week_start'], rolling_avg, color='#C00000', lw=2.2, linestyle='--', label='8-Wk Moving Avg')
    
    ax.set_title(f"{sku}: {sku_name} ({cat})", fontsize=11, fontweight='bold')
    ax.set_ylabel('Units Sold')
    ax.legend(loc='upper left')

plt.suptitle('Weekly Demand Trajectories & Seasonal Spikes (Top 4 Revenue SKUs)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 6. Supplier Lead Time Performance
Supplier fulfillment lead times are the foundational driver of safety stock and reorder points. Let's inspect supplier lead time means and standard deviations from `v_supplier_performance`.


In [7]:
supp_df = pd.read_sql_query('''
    SELECT * FROM v_supplier_performance;
''', conn)

conn.close()
supp_df


### Summary of Key Findings from EDA
1. **Catalog Pareto Concentration:** The top 20 SKUs represent over **75% of total company revenue**, validating our strategy to focus the forecasting and inventory optimization pipeline on these high-leverage items.
2. **Clear Seasonality & Trends:** Category-specific demand surges are pronounced (e.g. Q4 holiday rushes for Consumer Electronics and Home Goods, summer surges for Outdoor equipment).
3. **Supplier Variability:** Lead times range from an average of **7 days (1.0 week)** for local domestic suppliers to **28 days (4.0 weeks)** for overseas vendors.

*Next Step:* In **02_forecasting.ipynb**, we train time-series models with 12-week holdout validation to forecast demand for the upcoming quarter.
